# Noveria History — Kokoro Batch Narration
Use Chrome voice while editing, then run this notebook when you want the final Kokoro narration. Default voice: **bm_george** (British male).
This notebook works interactively and does not start a public tunnel/API.


In [ ]:
!pip -q install 'kokoro>=0.9.4' soundfile
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
from kokoro import KPipeline
from google.colab import files
import numpy as np, soundfile as sf, io, zipfile, json, re, os, base64, shutil, time
pipeline = KPipeline(lang_code='b')
VOICE = 'bm_george'  # bm_george, bm_daniel, bm_fable, bm_lewis
DEFAULT_SPEED = 1.0
print('Kokoro ready:', VOICE)


## Optional: connect your Google Drive
Set `USE_GOOGLE_DRIVE=True` if you want Colab to save a copy of the generated files into your Drive. Google handles the account authorization screen.


In [ ]:
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT = '/content/drive/MyDrive/MEP-Video-Maker/voices'
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    print('Drive output:', DRIVE_OUTPUT)


## Upload the MEP narration batch
In MEP Video Maker click **Download Colab Narration Batch**, then upload that JSON file here.


In [ ]:
uploaded = files.upload()
filename = next((name for name in uploaded if name.lower().endswith('.json')), None)
assert filename, 'Upload the narration-batch .json file from MEP Video Maker'
batch = json.loads(uploaded[filename].decode('utf-8'))
items = batch.get('shorts', batch if isinstance(batch, list) else [])
assert isinstance(items, list) and 1 <= len(items) <= 100, 'No Shorts found in the batch'
print('Shorts loaded:', len(items))


In [ ]:
def make_wav(text, voice, speed):
    pieces = []
    for _gs, _ps, audio in pipeline(text, voice=voice, speed=speed):
        if audio is None: continue
        if hasattr(audio, 'detach'): audio = audio.detach().cpu().numpy()
        pieces.append(np.asarray(audio, dtype=np.float32))
    if not pieces: raise RuntimeError('Kokoro returned no audio')
    audio = np.concatenate(pieces)
    wav = io.BytesIO()
    sf.write(wav, audio, 24000, format='WAV', subtype='PCM_16')
    return wav.getvalue(), len(audio) / 24000.0

pack = {'schema':'mep-ai-voice-pack-v1','engine':'Kokoro-82M','createdAt':time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),'items':[]}
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as archive:
    for idx, item in enumerate(items, 1):
        short_id = str(item.get('id') or item.get('shortId') or '')
        title = str(item.get('title') or f'short-{idx}')
        text = str(item.get('narration') or item.get('text') or '').strip()
        if not text:
            print('Skipping empty narration:', title)
            continue
        voice = str(item.get('voice') or VOICE)
        if not voice.startswith('bm_'): voice = VOICE
        speed = float(item.get('speed') or DEFAULT_SPEED)
        wav_bytes, duration = make_wav(text, voice, speed)
        safe = re.sub(r'[^A-Za-z0-9_-]+','-',title).strip('-')[:70] or f'short-{idx}'
        wav_name = f'{idx:02d}-{safe}.wav'
        archive.writestr(wav_name, wav_bytes)
        pack['items'].append({'shortId':short_id,'title':title,'voice':voice,'speed':speed,'duration':duration,'mime':'audio/wav','dataUrl':'data:audio/wav;base64,'+base64.b64encode(wav_bytes).decode('ascii')})
        print(f'{idx:02d} {title}: {duration:.1f}s · {voice}')

with open('mep_voice_pack.json','w',encoding='utf-8') as f:
    json.dump(pack,f)
with open('mep_voice_wavs.zip','wb') as f:
    f.write(zip_buffer.getvalue())
print('Generated', len(pack['items']), 'voice tracks')


In [ ]:
if USE_GOOGLE_DRIVE:
    shutil.copy2('mep_voice_pack.json', DRIVE_OUTPUT + '/mep_voice_pack.json')
    shutil.copy2('mep_voice_wavs.zip', DRIVE_OUTPUT + '/mep_voice_wavs.zip')
    print('Saved copies to Google Drive')
files.download('mep_voice_pack.json')
files.download('mep_voice_wavs.zip')


Back in MEP Video Maker, choose **Import Colab Voice Pack** and select `mep_voice_pack.json`. The app matches tracks to Shorts by ID/title and stores the WAV audio in the project.
